# Dalin Data Cleaning


In [22]:
import pandas as pd 
import numpy as np
import json

In [23]:
data = pd.read_json("/Users/dalintry/dso576/project/dataset/Appliances.jsonl", lines=True)
data.head(5)

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Work great,work great. use a new one every month,[],B01N0TQ0OH,B01N0TQ0OH,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2018-02-22 16:31:48.692,0,True
1,5,excellent product,Little on the thin side,[],B07DD2DMXB,B07DD37QPZ,AHWWLSPCJMALVHDDVSUGICL6RUCA,2022-10-02 21:41:03.446,0,True
2,5,Happy customer!,"Quick delivery, fixed the issue!",[],B082W3Z9YK,B082W3Z9YK,AHZIJGKEWRTAEOZ673G5B3SNXEGQ,2020-12-06 03:30:35.363,0,True
3,5,Amazing value,I wasn't sure whether these were worth it or n...,[],B078W2BJY8,B078W2BJY8,AFGUPTDFAWOHHL4LZDV27ERDNOYQ,2018-08-12 20:03:04.306,0,True
4,5,Dryer parts,Easy to install got the product expected to re...,[],B08C9LPCQV,B08C9LPCQV,AELFJFAXQERUSMTXJQ6SYFFRDWMA,2021-05-05 01:03:23.754,0,True


In [24]:
column_types = pd.DataFrame({"column": data.columns, 
                             "data_type": data.dtypes,
                             "non_null_count": data.notnull().sum()
                             ,"nulls_count": data.isnull().sum()})
column_types

,column,data_type,non_null_count,nulls_count
rating,rating,int64,2128605,0
title,title,str,2128605,0
text,text,str,2128605,0
images,images,object,2128605,0
asin,asin,str,2128605,0
parent_asin,parent_asin,str,2128605,0
user_id,user_id,str,2128605,0
timestamp,timestamp,datetime64[ms],2128605,0
helpful_vote,helpful_vote,int64,2128605,0
verified_purchase,verified_purchase,bool,2128605,0


In [25]:
# Describe the type each column SHOULD have. Edit this plan if your data dictionary says otherwise.
# pandas nullable types (Int64, Float64, boolean, string) keep missing values safely.
conversion_plan = {
    "rating": {"type": "Int64"},
    "title": {"type": "string"},
    "text": {"type": "string"},
    "images": {"type": "object"},  # lists belong in an object column
    "asin": {"type": "string"},
    "parent_asin": {"type": "string"},
    "user_id": {"type": "string"},
    "timestamp": {"type": "datetime64[ns]"},
    "helpful_vote": {"type": "Int64"},
    "verified_purchase": {"type": "boolean"},
}

conversion_plan_df = pd.DataFrame.from_dict(conversion_plan, orient="index")
conversion_plan_df


,type
rating,Int64
title,string
text,string
images,object
asin,string
parent_asin,string
user_id,string
timestamp,datetime64[ns]
helpful_vote,Int64
verified_purchase,boolean


## 4 Inspect columns and types
#### List incorrect types and your conversions. Count failed conversions and explain how you handled them.


In [26]:
# Columns to inspect and their intended conversions
type_conversions = {
    "rating": "numeric",
    "timestamp": "datetime",
    "helpful_vote": "numeric",
    "verified_purchase": "boolean"
}

conversion_results = []

for column, target_type in type_conversions.items():
    original = data[column]

    if target_type == "numeric":
        converted = pd.to_numeric(original, errors="coerce")

    elif target_type == "datetime":
        converted = pd.to_datetime(original, errors="coerce")

    elif target_type == "boolean":
        boolean_map = {
            "true": True, "false": False,
            "yes": True, "no": False,
            "1": True, "0": False
        }
        converted = (
            original.astype("string")
            .str.strip()
            .str.lower()
            .map(boolean_map)
            .astype("boolean")
        )

    # A failure is a non-null value that became null during conversion
    failed = original.notna() & converted.isna()

    conversion_results.append({
        "column": column,
        "original_type": str(original.dtype),
        "converted_type": str(converted.dtype),
        "failed_conversions": int(failed.sum()),
        "failed_examples": original[failed].drop_duplicates().head(5).tolist()
    })

    # Replace the original column with the converted column
    data[column] = converted

conversion_report = pd.DataFrame(conversion_results)
conversion_report

,column,original_type,converted_type,failed_conversions,failed_examples
0,rating,int64,int64,0,[]
1,timestamp,datetime64[ms],datetime64[ms],0,[]
2,helpful_vote,int64,int64,0,[]
3,verified_purchase,bool,boolean,0,[]


## 5 Standardize text
### Show text before and after cleaning. Keep meaningful differences and leading zeros in IDs.


In [27]:
text_columns = ["title", "text"]

comparison_results = []

for column in text_columns:
    before = data[column].astype("string")

    # Conservative cleaning:
    # - standardize Unicode
    # - collapse repeated spaces/tabs
    # - remove leading/trailing whitespace
    # - preserve capitalization, punctuation, and wording
    after = (
        before
        .str.normalize("NFC")
        .str.replace(r"[ \t]+", " ", regex=True)
        .str.strip()
    )

    changed = before.ne(after).fillna(False)

    comparison_results.append(
        pd.DataFrame({
            "column": column,
            "before": before[changed],
            "after": after[changed]
        })
    )

    data[column] = after

# Show examples that changed
text_cleaning_examples = pd.concat(comparison_results).head(20)
text_cleaning_examples

column = "text"

before = data[column].astype("string")
after = (
    before
    .str.normalize("NFC")
    .str.replace(r"[ \t]+", " ", regex=True)
    .str.strip()
)

pd.DataFrame({
    "before": before,
    "after": after
}).head(10)

,before,after
0,work great. use a new one every month,work great. use a new one every month
1,Little on the thin side,Little on the thin side
2,"Quick delivery, fixed the issue!","Quick delivery, fixed the issue!"
3,I wasn't sure whether these were worth it or n...,I wasn't sure whether these were worth it or n...
4,Easy to install got the product expected to re...,Easy to install got the product expected to re...
5,After buying this ice machine just 15 months a...,After buying this ice machine just 15 months a...
6,Not the best quality,Not the best quality
7,Part came quickly and fit my LG dryer. Thanks!,Part came quickly and fit my LG dryer. Thanks!
8,Always arrive in a fast manner. Descriptions o...,Always arrive in a fast manner. Descriptions o...
9,The company responded very quickly. Refunded p...,The company responded very quickly. Refunded p...


## 6 Investigate duplicates
### Count exact duplicate rows and repeated IDs separately. 

In [28]:
# Create a temporary version for duplicate checking.
# Convert list/dictionary columns to text so pandas can compare them.
duplicate_check = data.copy(deep=False)

for column in duplicate_check.columns:
    if duplicate_check[column].map(
        lambda value: isinstance(value, (list, dict))
    ).any():
        duplicate_check[column] = duplicate_check[column].map(
            lambda value: json.dumps(value, sort_keys=True)
            if isinstance(value, (list, dict))
            else value
        )

# Duplicate rows after the first occurrence
duplicate_rows = duplicate_check.duplicated().sum()

# All rows involved in duplication, including the first occurrence
rows_in_duplicate_groups = duplicate_check.duplicated(keep=False).sum()

print("Extra duplicate rows:", duplicate_rows)
print("All rows involved in duplicate groups:", rows_in_duplicate_groups)

Extra duplicate rows: 22656
All rows involved in duplicate groups: 41798


### Count repeated IDs:

In [29]:
id_columns = ["asin", "parent_asin", "user_id"]

repeated_id_results = []

for column in id_columns:
    values = data[column].dropna()

    repeated_id_results.append({
        "column": column,
        "unique_ids": values.nunique(),
        "ids_repeated": values[values.duplicated(keep=False)].nunique(),
        "extra_occurrences": values.duplicated().sum()
    })

repeated_id_summary = pd.DataFrame(repeated_id_results)
repeated_id_summary

,column,unique_ids,ids_repeated,extra_occurrences
0,asin,104237,67396,2024368
1,parent_asin,94319,60731,2034286
2,user_id,1755732,259221,372873


## 7 Handle missing data
### Report missing counts before and after cleaning for each affected column.

In [30]:
# Count missing values and calculate their percentage
missing_summary = pd.DataFrame({
    "missing_count": data.isna().sum(),
    "missing_percent": (data.isna().mean() * 100).round(2)
})

missing_summary = missing_summary.sort_values(
    "missing_count",
    ascending=False
)

missing_summary

,missing_count,missing_percent
rating,0,0.0
title,0,0.0
text,0,0.0
images,0,0.0
asin,0,0.0
parent_asin,0,0.0
user_id,0,0.0
timestamp,0,0.0
helpful_vote,0,0.0
verified_purchase,0,0.0


In [31]:
missing_summary[missing_summary["missing_count"] > 0]

,missing_count,missing_percent


## 8 Handle missing data
### Check invalid dates, impossible values, and unusual records. Explain what you corrected, kept, or flagged.

In [32]:
# Convert values temporarily for testing; this does not modify data
checked_dates = pd.to_datetime(data["timestamp"], errors="coerce")
checked_ratings = pd.to_numeric(data["rating"], errors="coerce")
checked_votes = pd.to_numeric(data["helpful_vote"], errors="coerce")

# Missing or invalid values
missing_values = data.isna()
invalid_dates = data["timestamp"].notna() & checked_dates.isna()

# Date range can be adjusted if needed
dates_too_old = checked_dates < pd.Timestamp("1995-01-01")
future_dates = checked_dates > pd.Timestamp.now()

invalid_ratings = (
    data["rating"].notna()
    & (checked_ratings.isna() | ~checked_ratings.between(1, 5))
)

invalid_votes = (
    data["helpful_vote"].notna()
    & (checked_votes.isna() | checked_votes.lt(0))
)

# Blank strings are not detected by isna()
text_columns = ["title", "text", "asin", "parent_asin", "user_id"]

blank_counts = {
    column: int(data[column].astype("string").str.strip().eq("").sum())
    for column in text_columns
}

suspicious_summary = pd.DataFrame({
    "issue": [
        "Missing values",
        "Blank text or IDs",
        "Invalid dates",
        "Dates before 1995",
        "Future dates",
        "Invalid ratings",
        "Invalid helpful votes"
    ],
    "count": [
        int(missing_values.sum().sum()),
        sum(blank_counts.values()),
        int(invalid_dates.sum()),
        int(dates_too_old.sum()),
        int(future_dates.sum()),
        int(invalid_ratings.sum()),
        int(invalid_votes.sum())
    ]
})

suspicious_summary

,issue,count
0,Missing values,0
1,Blank text or IDs,2065
2,Invalid dates,0
3,Dates before 1995,0
4,Future dates,0
5,Invalid ratings,0
6,Invalid helpful votes,0


## 9 Verify individual records
### Check five records, including difficult cases.


In [33]:
# Create masks for difficult cases
checked_dates = pd.to_datetime(data["timestamp"], errors="coerce")

missing_mask = data.isna().any(axis=1)

suspicious_date_mask = (
    (data["timestamp"].notna() & checked_dates.isna())
    | (checked_dates < pd.Timestamp("1995-01-01"))
    | (checked_dates > pd.Timestamp.now())
)

duplicate_mask = data.duplicated(
    subset=["user_id", "asin", "timestamp"],
    keep=False
)

# Store five distinct record indexes and the reason each was selected
selected_records = {}

def add_first_case(mask, reason):
    matching_indexes = data.index[mask]

    for index in matching_indexes:
        if index not in selected_records:
            selected_records[index] = reason
            return

add_first_case(missing_mask, "Contains missing data")
add_first_case(suspicious_date_mask, "Contains a suspicious date")
add_first_case(duplicate_mask, "Has a repeated review key")

# Record with the longest review text
text_lengths = data["text"].astype("string").str.len().fillna(0)
longest_text_index = text_lengths.idxmax()

if longest_text_index not in selected_records:
    selected_records[longest_text_index] = "Longest review text"

# Record with the highest helpful-vote count
vote_values = pd.to_numeric(data["helpful_vote"], errors="coerce")

if vote_values.notna().any():
    highest_vote_index = vote_values.idxmax()

    if highest_vote_index not in selected_records:
        selected_records[highest_vote_index] = "Highest helpful-vote count"

# If some difficult cases do not exist, fill the sample with other records
for index in data.index:
    if len(selected_records) == 5:
        break

    if index not in selected_records:
        selected_records[index] = "Additional record for manual review"

# Display the selected records
records_to_check = data.loc[list(selected_records)].copy()
records_to_check.insert(
    0,
    "case_reason",
    [selected_records[index] for index in records_to_check.index]
)

records_to_check

,case_reason,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
2873,Has a repeated review key,5,"Hey, it fixed my refrigerator which cost me a ...","Hey, it fixed my refrigerator which cost me a ...",[],B001DPJTLC,B001DPJTLC,AGPIVP6KGLV3RC4MFKELOSHUT76A,2016-08-03 03:30:24.000,0,True
1150178,Longest review text,1,broke within 6-7 months.,Do Not buy. We purchased this in July 2017. We...,[{'small_image_url': 'https://images-na.ssl-im...,B01BKDEGP0,B01BKDEGP0,AGEQQBCKVTSBLOWAWQXC3MVMNA5A,2018-01-15 18:15:54.463,0,False
104598,Highest helpful-vote count,5,A True Game Changer! Small Living Life Saver :-),[[VIDEOID:95037017fece0fa9597755118ff29a40]] A...,[],B01ALBMIEI,B01ALBMIEI,AFFKAD3TNCYHYVK4D3QR5KNIPKNQ,2016-04-10 19:36:51.000,5704,True
0,Additional record for manual review,5,Work great,work great. use a new one every month,[],B01N0TQ0OH,B01N0TQ0OH,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2018-02-22 16:31:48.692,0,True
1,Additional record for manual review,5,excellent product,Little on the thin side,[],B07DD2DMXB,B07DD37QPZ,AHWWLSPCJMALVHDDVSUGICL6RUCA,2022-10-02 21:41:03.446,0,True
